In [2]:
%pip install -r ../requirements.txt
print('Dependencies installed or already satisfied for Task 2 notebook.')

  Using cached pandas-2.1.4-cp311-cp311-win_amd64.whl.metadata (18 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached matplotlib-3.8.2-cp311-cp311-win_amd64.whl.metadata (5.9 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.4.0-1-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached sentence_transformers-2.7.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached torch-2.2.2-cp311-cp311-win_amd64.whl.metadata (26 kB)
  Using cached tokenizers-0.19.1-cp311-none-win_amd64.whl.metadata (6.9 kB)
  Using cached faiss_cpu-1.8.0-cp311-cp311-win_amd64.whl.metadata (3.8 kB)
  Using cached chromadb-0.5.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached langchain-0.2.0-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_community-0.2.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached langchain_huggingface-0.0.3-py3-none-any.whl.meta


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Task 2: Text Chunking, Embedding & Vector Store Indexing
**CrediTrust Financial — RAG Complaint Chatbot**

This notebook covers:
1. Stratified sampling (10 K–15 K complaints)
2. Text chunking with overlap
3. Sentence-transformer embeddings
4. ChromaDB vector store creation and verification

## 1. Setup

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.embed_and_index import (
    stratified_sample,
    chunk_text,
    build_chunks_dataframe,
    load_embedding_model,
    embed_chunks,
    build_chroma_store,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    EMBEDDING_MODEL,
)

plt.rcParams["figure.dpi"] = 110
print("Imports OK ✅")

ValueError: The onnxruntime python package is not installed. Please install it with `pip install onnxruntime`

## 2. Load Cleaned Data

In [ ]:
INPUT_PATH = "../data/filtered_complaints.csv"
STORE_PATH = "../vector_store/"
N_SAMPLE = 12_000

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df):,} rows")
df.head(3)

## 3. Stratified Sampling

We draw a **stratified random sample** of 12 000 complaints, preserving the
proportional distribution of product categories.  This ensures all four
product types are fairly represented in the development vector store.

In [ ]:
df_sample = stratified_sample(df, n=N_SAMPLE, stratify_col="product_category")

fig, ax = plt.subplots(figsize=(8, 4))
counts = df_sample["product_category"].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette="viridis", ax=ax)
ax.bar_label(ax.containers[0], fmt="%d", padding=3)
ax.set_title(f"Sample Distribution (n={N_SAMPLE:,})")
ax.set_xlabel("Product Category")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Text Chunking

**Strategy:** Character-level sliding window with `chunk_size=500` and
`chunk_overlap=50`.

**Rationale:**
- `all-MiniLM-L6-v2` has a 256-token limit (~500–700 chars).  500-char chunks
  stay comfortably within budget without aggressive truncation.
- A 50-char overlap (~10%) preserves sentence context at chunk boundaries,
  reducing the risk of splitting mid-sentence and losing semantic coherence.

In [ ]:
print(f"Chunk size  : {CHUNK_SIZE} characters")
print(f"Chunk overlap: {CHUNK_OVERLAP} characters")

example = df_sample["cleaned_narrative"].iloc[0]
example_chunks = chunk_text(example, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"\nExample narrative length : {len(example)} chars")
print(f"Number of chunks produced: {len(example_chunks)}")
for i, c in enumerate(example_chunks):
    print(f"\n  Chunk {i} ({len(c)} chars): {c[:120]}…")

In [ ]:
chunks_df = build_chunks_dataframe(df_sample)
print(f"\nTotal chunks: {len(chunks_df):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

chunks_per_complaint = chunks_df.groupby("complaint_id").size()
chunks_per_complaint.clip(upper=15).hist(bins=15, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Chunks per Complaint (capped at 15)")
axes[0].set_xlabel("Number of Chunks")
axes[0].set_ylabel("Frequency")

chunk_len = chunks_df["text"].str.len()
chunk_len.hist(bins=30, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Chunk Length Distribution (chars)")
axes[1].set_xlabel("Characters")

plt.tight_layout()
plt.show()

print(f"\nChunks per complaint — mean: {chunks_per_complaint.mean():.1f} | "
      f"median: {chunks_per_complaint.median():.0f} | "
      f"max: {chunks_per_complaint.max()}")

## 5. Embedding

**Model choice:** `sentence-transformers/all-MiniLM-L6-v2`

- **Speed:** ~14 000 sentences/second on CPU — feasible for 12 K+ chunks.
- **Size:** 80 MB model, 384-dimensional vectors — low memory overhead.
- **Quality:** Strong multilingual capability and good semantic clustering for
  financial complaint text, as evidenced by SBERT benchmarks.
- **Consistency:** The same model is provided in the pre-built vector store,
  ensuring embedding-space compatibility when switching to the full dataset.

In [ ]:
model = load_embedding_model(EMBEDDING_MODEL)
print(f"Model: {EMBEDDING_MODEL}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
embeddings = embed_chunks(chunks_df["text"].tolist(), model)
print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"L2 norm of first vector (should be ~1.0 if normalised): {np.linalg.norm(embeddings[0]):.4f}")

## 6. Vector Store Indexing

In [ ]:
collection = build_chroma_store(chunks_df, embeddings, STORE_PATH)
print(f"\nCollection document count: {collection.count():,}")

## 7. Verify Retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
retrieval_model = SentenceTransformer(EMBEDDING_MODEL)

test_query = "My credit card was charged twice for the same purchase"
query_vec = retrieval_model.encode(test_query, normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=[query_vec],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: '{test_query}'\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    print(f"[Result {i+1}]  dist={dist:.4f}  |  {meta.get('product_category','')}  |  {meta.get('issue','')}")
    print(f"  {doc[:250]}\n")

## 8. Save Chunk Metadata

In [ ]:
meta_path = os.path.join(STORE_PATH, "chunks_metadata.parquet")
chunks_df.drop(columns=["text"]).to_parquet(meta_path, index=False)
print(f"✅  Chunk metadata saved → {meta_path}")